In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re

def test_crawl_job(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "vi-VN,vi;q=0.9,en;q=0.8"
    }
    
    try:
        print(f"🔄 Đang tải: {url}")
        response = requests.get(url, headers=headers, timeout=15)
        
        if response.status_code != 200:
            print(f"❌ Lỗi HTTP: {response.status_code}")
            return None

        soup = BeautifulSoup(response.text, "html.parser")
        
        # --- PHẦN SỬA ĐỔI: Lấy Tên công việc ---
        job_title = "Không tìm thấy tên"
        title_tag = soup.find("h1", class_="job-detail__info--title")
        if title_tag:
            # Ưu tiên tìm thẻ a bên trong như bạn yêu cầu
            a_tag = title_tag.find("a")
            if a_tag:
                job_title = a_tag.get_text(strip=True)
            else:
                # Nếu không có thẻ a thì lấy text trực tiếp của h1
                job_title = title_tag.get_text(strip=True)
        # ---------------------------------------

        # Khởi tạo dict và đưa Tên công việc lên đầu
        job_info = {"Tên công việc": job_title}

        # --- Lấy các thông tin khác (Code cũ) ---
        output_lines = []
        
        # 1. Tên công ty
        company_label = soup.find("div", class_="company-name-label")
        if company_label and company_label.find("a"):
            output_lines.append(f"=== Tên công ty ===\n{company_label.find('a').get_text(strip=True)}")

        # 2. Thông tin header (Lương, địa điểm...)
        info_sections = soup.find_all("div", class_="job-detail__info--section")
        for section in info_sections:
            title_tag_sec = section.find("div", class_="job-detail__info--section-content-title")
            val_tag_sec = section.find("div", class_="job-detail__info--section-content-value")
            
            t_text = title_tag_sec.get_text(strip=True) if title_tag_sec else "Info"
            v_text = val_tag_sec.get_text(strip=True) if val_tag_sec else ""
            output_lines.append(f"=== {t_text} ===\n{v_text}")

        # 3. Nội dung mô tả (Mô tả, Yêu cầu, Quyền lợi...)
        job_desc_div = soup.find("div", class_=lambda c: c and "job-description" in c)
        if job_desc_div:
            desc_items = job_desc_div.find_all("div", class_="job-description__item")
            for item in desc_items:
                h3 = item.find("h3")
                title = h3.get_text(strip=True) if h3 else "Mục khác"
                content_div = item.find("div", class_="job-description__item--content")
                content = content_div.get_text(separator="\n", strip=True) if content_div else ""
                output_lines.append(f"=== {title} ===\n{content}")

        # Ghép data vào dict
        full_text = "\n".join(output_lines)
        pattern = r"===\s*(.*?)\s*===\s*([\s\S]*?)(?===|$)"
        matches = re.findall(pattern, full_text)
        
        for title, content in matches:
            job_info[title.strip()] = content.strip()
            
        return job_info

    except Exception as e:
        print(f"❌ Lỗi ngoại lệ: {e}")
        return None

# === CHẠY THỬ ===
test_url = "https://www.topcv.vn/viec-lam/thuc-tap-sinh-cong-nghe-thong-tin-co-luong/1977202.html?ta_source=ITJobs_LinkDetail"
result = test_crawl_job(test_url)

if result:
    print("\n✅ KẾT QUẢ JSON THU ĐƯỢC:")
    print(json.dumps(result, ensure_ascii=False, indent=4))
else:
    print("Không lấy được dữ liệu.")